# Pose-Only HMM for Social State Discovery

This notebook fits a pose-only Gaussian HMM to the home-cage social dataset and then asks whether the inferred latent states provide cleaner temporal structure than the sparse manual labels.

Workflow:
1. Load pose + DA data.
2. Preserve sequence boundaries by recording block and time-gap resets.
3. Z-score pose features within sequence.
4. Fit HMMs across 3 to 8 states.
5. Decode the latent state path.
6. Summarize state geometry, dwell time, transition probabilities, and overlap with manual labels.
7. Align DA to HMM state transitions.

Important design choice: **DA is not used as an HMM input feature**. It is only used downstream for transition-aligned analyses.

In [ ]:
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import logsumexp
from scipy.stats import zscore
from sklearn.cluster import KMeans

warnings.filterwarnings('ignore')
plt.style.use('default')

In [ ]:
DATA_PATH = Path('home_cage_pose_DA_all.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('home_cage_pose_DA.csv')

FEATURE_COLUMNS = [
    'distance_head_res__head_int',
    'angle_head_res__head_int_deg',
    'distance_head_res__hind_int',
    'angle_head_res__hind_int_deg',
    'distance_head_int__hind_res',
    'angle_head_int__hind_res_deg',
    'distance_hind_res__hind_int',
    'velocity_resident',
    'velocity_intruder',
]

COLUMN_RENAME_MAP = {
    'head-head_dist': 'distance_head_res__head_int',
    'res_head-int_head_angle': 'angle_head_res__head_int_deg',
    'res_head-int_hind_dist': 'distance_head_res__hind_int',
    'res_head-int_hind_angle': 'angle_head_res__hind_int_deg',
    'int_head-res_hind_dist': 'distance_head_int__hind_res',
    'int_head-res_hind_angle': 'angle_head_int__hind_res_deg',
    'hind-hind_dist': 'distance_hind_res__hind_int',
    'resident_velocity': 'velocity_resident',
    'intruder_velocity': 'velocity_intruder',
}

BRAIN_REGION_FILTER = None
MOUSE_FILTER = None
INTRUDER_FILTER = None

GAP_THRESHOLD_S = 0.75
MIN_SEQUENCE_LEN = 150
STATE_RANGE = range(3, 9)
N_INIT = 3
MAX_ITER = 40
RANDOM_STATE = 7
PRE_S = 3.0
POST_S = 5.0

In [ ]:
df = pd.read_csv(DATA_PATH).rename(columns=COLUMN_RENAME_MAP)

for col in FEATURE_COLUMNS + ['time_s', 'zscore_DA']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

if 'behavior_active' not in df.columns:
    df['behavior_active'] = np.nan

required_cols = ['time_s', 'mouse_identity', 'brain_region', 'intruder_identity'] + FEATURE_COLUMNS
missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

if BRAIN_REGION_FILTER is not None:
    df = df[df['brain_region'].isin(np.atleast_1d(BRAIN_REGION_FILTER))].copy()
if MOUSE_FILTER is not None:
    df = df[df['mouse_identity'].isin(np.atleast_1d(MOUSE_FILTER))].copy()
if INTRUDER_FILTER is not None:
    df = df[df['intruder_identity'].isin(np.atleast_1d(INTRUDER_FILTER))].copy()

df = df.sort_values(['mouse_identity', 'brain_region', 'intruder_identity', 'time_s']).reset_index(drop=True)
df['behavior_label'] = df['behavior_active'].fillna('').replace('', 'None')
df['base_block'] = (
    df['mouse_identity'].astype(str) + '|' +
    df['brain_region'].astype(str) + '|' +
    df['intruder_identity'].astype(str)
)

gap_breaks = df.groupby('base_block')['time_s'].diff().fillna(0).gt(GAP_THRESHOLD_S)
df['segment_in_block'] = gap_breaks.groupby(df['base_block']).cumsum().astype(int)
df['sequence_id'] = df['base_block'] + '|seg' + df['segment_in_block'].astype(str)

df = df.dropna(subset=FEATURE_COLUMNS).copy()
seq_lengths = df.groupby('sequence_id').size()
valid_sequences = seq_lengths[seq_lengths >= MIN_SEQUENCE_LEN].index
df = df[df['sequence_id'].isin(valid_sequences)].copy()

summary = pd.DataFrame({
    'rows': [len(df)],
    'mice': [df['mouse_identity'].nunique()],
    'brain_regions': [df['brain_region'].nunique()],
    'intruders': [df['intruder_identity'].nunique()],
    'base_blocks': [df['base_block'].nunique()],
    'sequences_after_gap_split': [df['sequence_id'].nunique()],
})

display(summary)
display(df['behavior_label'].value_counts(dropna=False).head(10).to_frame('frames'))

In [ ]:
def zscore_within_sequence(frame, feature_cols):
    parts = []
    for seq_id, seq_df in frame.groupby('sequence_id', sort=False):
        seq = seq_df.copy()
        values = seq[feature_cols].to_numpy(dtype=float)
        mean = np.nanmean(values, axis=0, keepdims=True)
        std = np.nanstd(values, axis=0, keepdims=True)
        std[std < 1e-6] = 1.0
        seq[[f'{c}_z' for c in feature_cols]] = (values - mean) / std
        parts.append(seq)
    return pd.concat(parts, axis=0).reset_index(drop=True)

z_df = zscore_within_sequence(df, FEATURE_COLUMNS)
Z_FEATURES = [f'{c}_z' for c in FEATURE_COLUMNS]

X = z_df[Z_FEATURES].to_numpy(dtype=float)
lengths = z_df.groupby('sequence_id', sort=False).size().tolist()
sequence_ids = z_df['sequence_id'].drop_duplicates().tolist()

dt = float(z_df.groupby('sequence_id')['time_s'].diff().median())
if not np.isfinite(dt) or dt <= 0:
    dt = 0.1

pre_frames = int(round(PRE_S / dt))
post_frames = int(round(POST_S / dt))

X.shape, len(lengths), dt

In [ ]:
class DiagonalGaussianHMM:
    def __init__(self, n_states, n_iter=30, tol=1e-3, min_covar=1e-3, random_state=0):
        self.n_states = n_states
        self.n_iter = n_iter
        self.tol = tol
        self.min_covar = min_covar
        self.random_state = random_state

    def _split(self, X, lengths):
        out = []
        start = 0
        for L in lengths:
            out.append(X[start:start + L])
            start += L
        return out

    def _initialize(self, X):
        rng = np.random.default_rng(self.random_state)
        km = KMeans(n_clusters=self.n_states, n_init=10, random_state=self.random_state)
        labels = km.fit_predict(X)
        self.means_ = km.cluster_centers_.copy()
        self.covars_ = np.zeros((self.n_states, X.shape[1]), dtype=float)
        for k in range(self.n_states):
            members = X[labels == k]
            if len(members) == 0:
                pick = X[rng.integers(0, len(X), size=max(10, len(X) // self.n_states))]
                members = pick
                self.means_[k] = pick.mean(axis=0)
            var = members.var(axis=0) + self.min_covar
            self.covars_[k] = np.maximum(var, self.min_covar)
        self.startprob_ = np.full(self.n_states, 1.0 / self.n_states)
        trans = np.full((self.n_states, self.n_states), 1.0 / self.n_states)
        trans += np.eye(self.n_states) * 2.0
        self.transmat_ = trans / trans.sum(axis=1, keepdims=True)

    def _log_emission(self, X):
        diff = X[:, None, :] - self.means_[None, :, :]
        log_det = np.sum(np.log(self.covars_), axis=1)
        maha = np.sum((diff ** 2) / self.covars_[None, :, :], axis=2)
        return -0.5 * (X.shape[1] * np.log(2 * np.pi) + log_det[None, :] + maha)

    def _forward_backward(self, X):
        log_B = self._log_emission(X)
        log_start = np.log(np.clip(self.startprob_, 1e-12, None))
        log_trans = np.log(np.clip(self.transmat_, 1e-12, None))

        T = len(X)
        alpha = np.empty((T, self.n_states))
        alpha[0] = log_start + log_B[0]
        for t in range(1, T):
            alpha[t] = log_B[t] + logsumexp(alpha[t - 1][:, None] + log_trans, axis=0)

        beta = np.zeros((T, self.n_states))
        for t in range(T - 2, -1, -1):
            beta[t] = logsumexp(log_trans + log_B[t + 1][None, :] + beta[t + 1][None, :], axis=1)

        loglik = float(logsumexp(alpha[-1]))
        log_gamma = alpha + beta - loglik
        gamma = np.exp(log_gamma)

        xi_sum = np.zeros((self.n_states, self.n_states), dtype=float)
        for t in range(T - 1):
            log_xi_t = (
                alpha[t][:, None] +
                log_trans +
                log_B[t + 1][None, :] +
                beta[t + 1][None, :] -
                loglik
            )
            xi_sum += np.exp(log_xi_t)
        return loglik, gamma, xi_sum

    def fit(self, X, lengths):
        self._initialize(X)
        sequences = self._split(X, lengths)
        prev_loglik = -np.inf
        self.history_ = []

        for _ in range(self.n_iter):
            total_loglik = 0.0
            start_counts = np.zeros(self.n_states, dtype=float)
            trans_counts = np.zeros((self.n_states, self.n_states), dtype=float)
            gamma_sum = np.zeros(self.n_states, dtype=float)
            gamma_obs = np.zeros((self.n_states, X.shape[1]), dtype=float)
            gamma_obs2 = np.zeros((self.n_states, X.shape[1]), dtype=float)

            for seq in sequences:
                loglik, gamma, xi_sum = self._forward_backward(seq)
                total_loglik += loglik
                start_counts += gamma[0]
                trans_counts += xi_sum
                gamma_sum += gamma.sum(axis=0)
                gamma_obs += gamma.T @ seq
                gamma_obs2 += gamma.T @ (seq ** 2)

            self.startprob_ = np.clip(start_counts / start_counts.sum(), 1e-12, None)
            self.startprob_ /= self.startprob_.sum()

            trans_counts += 1e-6
            self.transmat_ = trans_counts / trans_counts.sum(axis=1, keepdims=True)

            gamma_sum_safe = np.clip(gamma_sum[:, None], 1e-12, None)
            self.means_ = gamma_obs / gamma_sum_safe
            second_moment = gamma_obs2 / gamma_sum_safe
            self.covars_ = np.maximum(second_moment - self.means_ ** 2, self.min_covar)

            self.history_.append(total_loglik)
            if abs(total_loglik - prev_loglik) < self.tol:
                break
            prev_loglik = total_loglik

        self.loglik_ = self.score(X, lengths)
        return self

    def score(self, X, lengths):
        total = 0.0
        for seq in self._split(X, lengths):
            total += self._forward_backward(seq)[0]
        return float(total)

    def bic(self, X, lengths):
        n_features = X.shape[1]
        n_params = (self.n_states - 1) + self.n_states * (self.n_states - 1) + 2 * self.n_states * n_features
        return -2 * self.score(X, lengths) + n_params * np.log(len(X))

    def predict(self, X, lengths):
        paths = []
        log_start = np.log(np.clip(self.startprob_, 1e-12, None))
        log_trans = np.log(np.clip(self.transmat_, 1e-12, None))
        for seq in self._split(X, lengths):
            log_B = self._log_emission(seq)
            T = len(seq)
            delta = np.empty((T, self.n_states))
            psi = np.empty((T, self.n_states), dtype=int)
            delta[0] = log_start + log_B[0]
            psi[0] = 0
            for t in range(1, T):
                scores = delta[t - 1][:, None] + log_trans
                psi[t] = np.argmax(scores, axis=0)
                delta[t] = log_B[t] + np.max(scores, axis=0)
            states = np.empty(T, dtype=int)
            states[-1] = np.argmax(delta[-1])
            for t in range(T - 2, -1, -1):
                states[t] = psi[t + 1, states[t + 1]]
            paths.append(states)
        return np.concatenate(paths)


def fit_hmm_grid(X, lengths, state_range, n_init=3, n_iter=30, base_seed=0):
    rows = []
    best_models = {}
    for k in state_range:
        best_model = None
        best_loglik = -np.inf
        for run in range(n_init):
            model = DiagonalGaussianHMM(
                n_states=k,
                n_iter=n_iter,
                random_state=base_seed + 100 * k + run,
            ).fit(X, lengths)
            if model.loglik_ > best_loglik:
                best_loglik = model.loglik_
                best_model = model
        best_models[k] = best_model
        rows.append({
            'n_states': k,
            'loglik': best_model.loglik_,
            'bic': best_model.bic(X, lengths),
            'iterations': len(best_model.history_),
        })
    return pd.DataFrame(rows).sort_values('n_states'), best_models

In [ ]:
model_selection, models = fit_hmm_grid(
    X,
    lengths,
    state_range=STATE_RANGE,
    n_init=N_INIT,
    n_iter=MAX_ITER,
    base_seed=RANDOM_STATE,
)

display(model_selection)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(model_selection['n_states'], model_selection['loglik'], marker='o')
axes[0].set_title('Model Log Likelihood')
axes[0].set_xlabel('Number of states')
axes[0].set_ylabel('Log likelihood')

axes[1].plot(model_selection['n_states'], model_selection['bic'], marker='o', color='tab:red')
axes[1].set_title('Model BIC')
axes[1].set_xlabel('Number of states')
axes[1].set_ylabel('BIC')
plt.tight_layout()

BEST_K = int(model_selection.sort_values('bic').iloc[0]['n_states'])
best_model = models[BEST_K]
print(f'Selected BEST_K = {BEST_K}')

In [ ]:
z_df = z_df.copy()
z_df['hmm_state'] = best_model.predict(X, lengths)

distance_cols = [
    'distance_head_res__head_int',
    'distance_head_res__hind_int',
    'distance_head_int__hind_res',
    'distance_hind_res__hind_int',
]
velocity_cols = ['velocity_resident', 'velocity_intruder']

state_feature_summary = z_df.groupby('hmm_state')[FEATURE_COLUMNS + ['zscore_DA']].mean()
state_counts = z_df['hmm_state'].value_counts().sort_index().rename('n_frames')

def dwell_table(frame):
    rows = []
    for seq_id, seq_df in frame.groupby('sequence_id', sort=False):
        states = seq_df['hmm_state'].to_numpy()
        times = seq_df['time_s'].to_numpy()
        start = 0
        for idx in range(1, len(seq_df) + 1):
            changed = idx == len(seq_df) or states[idx] != states[start]
            if changed:
                rows.append({
                    'sequence_id': seq_id,
                    'hmm_state': int(states[start]),
                    'n_frames': idx - start,
                    'duration_s': float(times[idx - 1] - times[start] + dt),
                })
                start = idx
    return pd.DataFrame(rows)

dwell_df = dwell_table(z_df)
state_dwell = dwell_df.groupby('hmm_state')[['n_frames', 'duration_s']].median().rename(
    columns={'n_frames': 'median_dwell_frames', 'duration_s': 'median_dwell_s'}
)

transition_counts = np.zeros((BEST_K, BEST_K), dtype=int)
for _, seq_df in z_df.groupby('sequence_id', sort=False):
    states = seq_df['hmm_state'].to_numpy()
    for a, b in zip(states[:-1], states[1:]):
        transition_counts[a, b] += 1
transition_probs = transition_counts / np.clip(transition_counts.sum(axis=1, keepdims=True), 1, None)

overview = pd.concat([state_counts, state_dwell], axis=1)
display(overview)
display(state_feature_summary.round(2))
pd.DataFrame(transition_probs).round(3)

In [ ]:
heatmap_data = state_feature_summary[FEATURE_COLUMNS]

fig, ax = plt.subplots(figsize=(12, 4 + 0.35 * BEST_K))
im = ax.imshow(heatmap_data.to_numpy(), aspect='auto', cmap='coolwarm')
ax.set_yticks(np.arange(BEST_K))
ax.set_yticklabels([f'State {s}' for s in heatmap_data.index])
ax.set_xticks(np.arange(len(FEATURE_COLUMNS)))
ax.set_xticklabels(FEATURE_COLUMNS, rotation=60, ha='right')
ax.set_title('Mean Raw Pose Features by HMM State')
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
plt.tight_layout()

In [ ]:
state_by_label = pd.crosstab(z_df['hmm_state'], z_df['behavior_label'])
state_by_label_frac = state_by_label.div(state_by_label.sum(axis=1), axis=0)
label_by_state_frac = state_by_label.div(state_by_label.sum(axis=0), axis=1)

print('Fraction of each HMM state occupied by each manual label:')
display(state_by_label_frac.round(3))

print('Fraction of each manual label captured by each HMM state:')
display(label_by_state_frac.round(3))

In [ ]:
mean_dist_by_state = state_feature_summary[distance_cols].mean(axis=1)
baseline_state = int(mean_dist_by_state.idxmax())
face_like_state = int(state_feature_summary['distance_head_res__head_int'].idxmin())
rear_like_state = int(state_feature_summary['distance_head_res__hind_int'].idxmin())
high_motion_state = int(state_feature_summary[velocity_cols].mean(axis=1).idxmax())

suggestions = pd.Series({
    'baseline_state_suggestion': baseline_state,
    'face_like_state_suggestion': face_like_state,
    'rear_like_state_suggestion': rear_like_state,
    'high_motion_state_suggestion': high_motion_state,
})
display(suggestions.to_frame('state'))

# Override these after inspecting the state summaries above if needed.
BASELINE_STATE = baseline_state
FACE_STATE = face_like_state
REAR_STATE = rear_like_state

In [ ]:
def extract_transition_events(frame, source_state=None, target_state=None, pre_frames=30, post_frames=50):
    events = []
    for seq_id, seq_df in frame.groupby('sequence_id', sort=False):
        seq_df = seq_df.reset_index(drop=True)
        states = seq_df['hmm_state'].to_numpy()
        da = seq_df['zscore_DA'].to_numpy(dtype=float)
        time_s = seq_df['time_s'].to_numpy(dtype=float)
        for idx in range(1, len(seq_df)):
            prev_state = int(states[idx - 1])
            curr_state = int(states[idx])
            if prev_state == curr_state:
                continue
            if source_state is not None and prev_state != source_state:
                continue
            if target_state is not None and curr_state != target_state:
                continue
            start = idx - pre_frames
            stop = idx + post_frames + 1
            if start < 0 or stop > len(seq_df):
                continue
            trace = da[start:stop]
            rel_time = time_s[start:stop] - time_s[idx]
            post_mask = rel_time >= 0
            peak = np.nanmax(trace[post_mask])
            peak_latency = rel_time[post_mask][np.nanargmax(trace[post_mask])]
            auc = np.trapz(trace[post_mask], rel_time[post_mask])
            pre_mask = rel_time < 0
            if pre_mask.sum() >= 2:
                pre_slope = np.polyfit(rel_time[pre_mask], trace[pre_mask], 1)[0]
            else:
                pre_slope = np.nan
            events.append({
                'sequence_id': seq_id,
                'entry_index': idx,
                'source_state': prev_state,
                'target_state': curr_state,
                'transition_label': f'{prev_state}->{curr_state}',
                'entry_time_s': time_s[idx],
                'trace': trace,
                'rel_time': rel_time,
                'peak': float(peak),
                'peak_latency_s': float(peak_latency),
                'auc_post': float(auc),
                'pre_slope': float(pre_slope),
            })
    return pd.DataFrame(events)


def plot_transition_average(events_df, title):
    if events_df.empty:
        print(f'No events for {title}')
        return
    traces = np.stack(events_df['trace'].to_numpy())
    rel_time = events_df['rel_time'].iloc[0]
    mean_trace = np.nanmean(traces, axis=0)
    sem_trace = np.nanstd(traces, axis=0, ddof=1) / math.sqrt(len(traces)) if len(traces) > 1 else np.zeros_like(mean_trace)

    plt.figure(figsize=(6, 4))
    plt.plot(rel_time, mean_trace, color='black', lw=2)
    plt.fill_between(rel_time, mean_trace - sem_trace, mean_trace + sem_trace, color='gray', alpha=0.3)
    plt.axvline(0, color='tab:red', ls='--', lw=1)
    plt.xlabel('Time from HMM state entry (s)')
    plt.ylabel('z-scored DA')
    plt.title(f'{title} (n={len(events_df)})')
    plt.tight_layout()

In [ ]:
transition_specs = [
    (BASELINE_STATE, FACE_STATE, 'baseline -> face-like'),
    (BASELINE_STATE, REAR_STATE, 'baseline -> rear-like'),
    (FACE_STATE, REAR_STATE, 'face-like -> rear-like'),
    (REAR_STATE, BASELINE_STATE, 'rear-like -> baseline'),
]

transition_tables = []
for source_state, target_state, label in transition_specs:
    events = extract_transition_events(
        z_df,
        source_state=source_state,
        target_state=target_state,
        pre_frames=pre_frames,
        post_frames=post_frames,
    )
    if not events.empty:
        summary = events[['peak', 'peak_latency_s', 'auc_post', 'pre_slope']].agg(['mean', 'median', 'count']).T
        summary['transition'] = label
        transition_tables.append(summary.reset_index().rename(columns={'index': 'metric'}))
    plot_transition_average(events, label)

if transition_tables:
    transition_summary = pd.concat(transition_tables, ignore_index=True)
    display(transition_summary)
else:
    print('No transition-aligned DA windows were extracted for the current state mapping.')

## Notes for interpretation

- Expect the HMM to discover latent dynamical states, not perfect semantic copies of manual labels.
- A useful result is often that one manual category spans multiple HMM states or that unlabeled substates emerge between manual onset and close contact.
- If the decoded states are too jittery, the next upgrade would be an autoregressive HMM rather than feeding DA into the emission model.
- If you want to focus only on manually labeled interaction periods, subset `df` before z-scoring with something like `df = df[df['behavior_label'] != 'None'].copy()`, but the cleaner first pass is usually to keep the full block.